In [23]:
"""
pctc_hardware_pipeline.py
=========================
Runs the PCTC m=1 post-selection pipeline on Quantinuum H2-1 (hardware).

Submits each circuit as a separate individual execute job (no batching),
which is the most reliable approach across all Nexus account tiers.

Pipeline
--------
1.  Build 6 circuit variants:
      tomo_Z, tomo_X, tomo_Y   — basis rotations on output qubit
      shadow_I, shadow_H, shadow_SH  — Clifford pre-rotations (shadows)

2.  For each circuit:
      a. Convert to pytket
      b. Compile on H2-1E (free)
      c. Execute compiled circuit on H2-1 with N_SHOTS shots

3.  Post-select on Bell outcome (crR=0, crG=0).

4.  Tomography fidelity: reconstruct rho from <X>, <Y>, <Z>.
    F_tomo = <psi_M|rho|psi_M>

5.  Classical shadows fidelity:
    F_shadow = mean(3|<b|U|psi>|^2 - 1)

6.  Bootstrap 95% CI for both (N_BOOT=1000).

7.  Save pctc_hardware_results.json + fig_pctc_hardware.pdf/png.
"""

import time
import json
import random
from collections import Counter
from math import pi, sqrt, cos, sin

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.circuit import OpType
import qnexus as qnx

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════
PROJECT_NAME    = "CTCs"
BACKEND_COMPILE = "H2-1E"
BACKEND_EXECUTE = "H2-1"
OPT_LEVEL       = 3
N_SHOTS         = 5000
N_BOOT          = 1000
# MAX_COST        = 200.0      # HQC cap per execute job (required by Helios-1)
RESULTS_FILE    = "pctc_hardware_results.json"

THETA  = 2.5349076035276403
VARPHI = 2.0022404587009195

SHADOW_GATES = ["I", "H", "SH"]

TWO_Q_TYPES = {OpType.ZZMax, OpType.ZZPhase, OpType.PhasedISWAP,
               OpType.ISWAP, OpType.CX, OpType.CZ, OpType.TK2}

# ══════════════════════════════════════════════════════════════════════════════
#  TARGET STATE
# ══════════════════════════════════════════════════════════════════════════════
def target_state_vector():
    return np.array([cos(THETA/2),
                     np.exp(1j*VARPHI)*sin(THETA/2)], dtype=complex)

def target_density_matrix():
    psi = target_state_vector()
    return np.outer(psi, psi.conj())

def target_bloch():
    rho = target_density_matrix()
    X = np.array([[0,1],[1,0]]); Y = np.array([[0,-1j],[1j,0]]); Z = np.array([[1,0],[0,-1]])
    return (float(np.real(np.trace(rho@X))),
            float(np.real(np.trace(rho@Y))),
            float(np.real(np.trace(rho@Z))))

SHADOW_UNITARIES = {
    "I":   np.eye(2, dtype=complex),
    "H":   np.array([[1,1],[1,-1]])/sqrt(2),
    "SH":  np.array([[1,1],[1j,-1j]])/sqrt(2),
    "X":   np.array([[0,1],[1,0]], dtype=complex),
    "HX":  np.array([[1,-1],[1,1]])/sqrt(2),
    "SHX": np.array([[1,-1j],[1j,-1]])/sqrt(2),
}

# ══════════════════════════════════════════════════════════════════════════════
#  CIRCUIT BUILDERS
# ══════════════════════════════════════════════════════════════════════════════
def _base_circuit():
    C=QuantumRegister(1,'qc'); E=QuantumRegister(1,'qe')
    R=QuantumRegister(1,'qr'); G=QuantumRegister(1,'qg')
    M=QuantumRegister(1,'qm'); A=QuantumRegister(1,'qa')
    Y=QuantumRegister(1,'qy')
    crR=ClassicalRegister(1,'crr'); crG=ClassicalRegister(1,'crg')
    crC=ClassicalRegister(1,'crc')
    qc=QuantumCircuit(C,E,R,G,M,A,Y,crR,crG,crC)
    qc.u(THETA,VARPHI,0.0,M[0])
    qc.swap(C[0],M[0]); qc.barrier()
    qc.h(E[0]); qc.cx(E[0],M[0])
    qc.h(R[0]); qc.cx(R[0],G[0])
    qc.h(A[0]); qc.cx(A[0],Y[0]); qc.barrier()
    qc.cz(C[0],R[0]); qc.cz(E[0],R[0]); qc.cz(C[0],E[0])
    qc.h(C[0]); qc.h(E[0]); qc.h(R[0])
    qc.cz(C[0],R[0]); qc.cz(C[0],E[0]); qc.cz(E[0],R[0]); qc.barrier()
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()
    qc.cx(R[0],G[0]); qc.h(R[0])
    qc.measure(R[0],crR[0]); qc.measure(G[0],crG[0]); qc.barrier()
    qc.swap(C[0],Y[0])
    qc.u(THETA,VARPHI,0.0,C[0]).inverse(); qc.barrier()
    return qc, C, crC

def build_circuit(label):
    qc, C, crC = _base_circuit()
    if   label == "X":   qc.h(C[0])
    elif label == "Y":   qc.sdg(C[0]); qc.h(C[0])
    elif label == "SH":  qc.h(C[0]); qc.s(C[0])
    elif label == "HX":  qc.x(C[0]); qc.h(C[0])
    elif label == "SHX": qc.x(C[0]); qc.h(C[0]); qc.s(C[0])
    # Z and I: no rotation needed
    qc.measure(C[0], crC[0])
    qc.name = f"pctc_{label}"
    return qc

# ══════════════════════════════════════════════════════════════════════════════
#  HQC COST ESTIMATOR
# ══════════════════════════════════════════════════════════════════════════════
def estimate_hqc(compiled_tk, n_shots):
    cmds  = compiled_tk.get_commands()
    n2q   = sum(1 for c in cmds if c.op.type in TWO_Q_TYPES)
    n1q   = sum(1 for c in cmds if len(c.args)==1
                and c.op.type not in (OpType.Measure, OpType.Reset))
    n_mid = 2  # crR and crG mid-circuit measurements
    hqc   = (n1q * 0.003 + n2q * 1.0 + n_mid * 10.0) * n_shots / 1000
    return hqc, n1q, n2q, n_mid

# ══════════════════════════════════════════════════════════════════════════════
#  NEXUS HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def poll_job(job, label="job"):
    print(f"  Polling {label}", end="", flush=True)
    while True:
        status = qnx.jobs.status(job)
        val    = status.status.value
        if val in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        print(".", end="", flush=True)
        time.sleep(15)
    print(f" {val}")
    if val != "COMPLETED":
        for attr in ["error_message", "error_detail", "detail", "message"]:
            v = getattr(status, attr, None)
            if v and v != "Circuit has encountered an error.":
                print(f"  {attr}: {v}")
        raise RuntimeError(f"{label} failed: {val}")
    return status


def compile_and_execute(qc, project, name, n_shots):
    """
    Full pipeline for a single circuit:
      1. Convert to pytket
      2. Upload logical circuit
      3. Compile on H2-1E → download compiled pytket circuit
      4. Upload compiled circuit separately
      5. Execute compiled circuit on H2-1
    Returns (result, compiled_tk)
    """
    tk = qiskit_to_tk(qc)

    # Step 1: upload logical
    logical_ref = qnx.circuits.upload(
        name=f"{name}_logical",
        circuit=tk,
        project=project,
    )

    # Step 2: compile on emulator
    compile_job = qnx.start_compile_job(
        programs=logical_ref,
        backend_config=qnx.QuantinuumConfig(device_name=BACKEND_COMPILE),
        optimisation_level=OPT_LEVEL,
        name=f"{name}_compile",
        project=project,
    )
    poll_job(compile_job, f"compile:{name}")
    compiled_tk = qnx.jobs.results(compile_job)[0].get_output().download_circuit()

    # Step 3: upload compiled circuit as a fresh ref
    compiled_ref = qnx.circuits.upload(
        name=f"{name}_compiled",
        circuit=compiled_tk,
        project=project,
    )

    # Step 4: execute compiled circuit on hardware
    execute_job = qnx.start_execute_job(
        programs=compiled_ref,          # single ref, not a list
        backend_config=qnx.QuantinuumConfig(device_name=BACKEND_EXECUTE, max_cost=MAX_COST),
        n_shots=n_shots,
        name=f"{name}_execute",
        project=project,
    )
    poll_job(execute_job, f"execute:{name}")

    result = qnx.jobs.results(execute_job)[0]
    return result, compiled_tk


def extract_bitstrings(result):
    counts     = result.get_counts()
    bitstrings = []
    for outcome, count in counts.items():
        def _bit(x): return int(x[0]) if isinstance(x, (list,tuple)) else int(x)
        crr = _bit(outcome[0]); crg = _bit(outcome[1]); crc = _bit(outcome[2])
        bitstrings.extend([(crr, crg, crc)] * int(count))
    return bitstrings


def post_select(bitstrings, label=""):
    selected = [b for b in bitstrings if b[0]==0 and b[1]==0]
    pct = 100*len(selected)/len(bitstrings) if bitstrings else 0
    print(f"    {label}  post-select: {len(selected)}/{len(bitstrings)} ({pct:.1f}%)")
    return selected

# ══════════════════════════════════════════════════════════════════════════════
#  TOMOGRAPHY FIDELITY
# ══════════════════════════════════════════════════════════════════════════════
def pauli_exp(selected, bit_idx=2):
    if not selected: return 0.0
    return float(np.mean([1-2*b[bit_idx] for b in selected]))

def tomography_fidelity(sel_Z, sel_X, sel_Y):
    rz,rx,ry = pauli_exp(sel_Z), pauli_exp(sel_X), pauli_exp(sel_Y)
    I=np.eye(2); X=np.array([[0,1],[1,0]]); Y=np.array([[0,-1j],[1j,0]]); Z=np.array([[1,0],[0,-1]])
    rho = (I + rx*X + ry*Y + rz*Z)/2
    eigvals,eigvecs = np.linalg.eigh(rho)
    eigvals = np.clip(eigvals,0,1); eigvals /= eigvals.sum()
    rho = eigvecs @ np.diag(eigvals) @ eigvecs.conj().T
    F   = float(np.real(np.trace(target_density_matrix() @ rho)))
    return F, rho, (rx,ry,rz)

def bootstrap_tomo(sel_Z, sel_X, sel_Y):
    Fs = []
    for _ in range(N_BOOT):
        bZ=[random.choice(sel_Z) for _ in range(max(1,len(sel_Z)))]
        bX=[random.choice(sel_X) for _ in range(max(1,len(sel_X)))]
        bY=[random.choice(sel_Y) for _ in range(max(1,len(sel_Y)))]
        Fs.append(tomography_fidelity(bZ,bX,bY)[0])
    Fs=np.array(Fs)
    return float(np.percentile(Fs,2.5)), float(np.percentile(Fs,97.5)), float(np.std(Fs))

# ══════════════════════════════════════════════════════════════════════════════
#  CLASSICAL SHADOWS FIDELITY
# ══════════════════════════════════════════════════════════════════════════════
def shadow_fidelity_estimator(shadow_data):
    psi = target_state_vector()
    ests = []
    for gate_label, bit in shadow_data:
        U     = SHADOW_UNITARIES[gate_label]
        b_vec = np.array([1.,0.]) if bit==0 else np.array([0.,1.])
        amp   = b_vec.conj() @ (U @ psi)
        ests.append(float(np.real(3*abs(amp)**2 - 1)))
    return float(np.mean(ests)) if ests else 0.0

def bootstrap_shadows(shadow_data):
    if not shadow_data: return 0.0,0.0,0.0
    Fs = [shadow_fidelity_estimator(
            [random.choice(shadow_data) for _ in range(len(shadow_data))])
          for _ in range(N_BOOT)]
    Fs=np.array(Fs)
    return float(np.percentile(Fs,2.5)), float(np.percentile(Fs,97.5)), float(np.std(Fs))

# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS FIGURE
# ══════════════════════════════════════════════════════════════════════════════
def plot_results(results):
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family":"serif","font.size":11,
                         "axes.linewidth":0.8,"xtick.direction":"in","ytick.direction":"in"})
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        r"PCTC $m=1$ on H2-1 (hardware): Output Fidelity $F(\rho_Y,\,\rho_M)$"
        f"\nPost-selected on Bell outcome 00  "
        f"({results.get('n_sel_Z','?')} / {results.get('n_raw_Z','?')} shots kept per basis)",
        fontsize=12, fontweight="bold", y=1.03)

    F_tomo=results["F_tomo"]; F_shad=results["F_shadows"]
    ci_t=results["F_tomo_ci"]; ci_s=results["F_shadows_ci"]

    ax=axes[0]
    fids=[F_tomo,F_shad]; ci_lo=[ci_t[0],ci_s[0]]; ci_hi=[ci_t[1],ci_s[1]]
    colors=["#4c72b0","#dd8452"]; x=np.arange(2)
    bars=ax.bar(x,fids,0.45,color=colors,alpha=0.88,edgecolor="white",linewidth=0.7)
    ax.errorbar(x,fids,
                yerr=[[f-lo for f,lo in zip(fids,ci_lo)],[hi-f for f,hi in zip(fids,ci_hi)]],
                fmt="none",color="black",capsize=6,capthick=1.5,lw=1.5)
    for bar,f in zip(bars,fids):
        ax.text(bar.get_x()+bar.get_width()/2, f+0.015, f"{f:.4f}",
                ha="center",va="bottom",fontsize=11,fontweight="bold")
    ax.axhline(1.0,color="green",lw=1.2,ls="--",alpha=0.5,label="$F=1$ (ideal)")
    ax.set_xticks(x); ax.set_xticklabels(["State\nTomography","Classical\nShadows"],fontsize=12)
    ax.set_ylabel(r"Fidelity  $F(\rho_Y,\,\rho_M)$",fontsize=12)
    ax.set_ylim(0,1.15)
    ax.set_title("Fidelity Estimates with 95% CI",fontsize=11,fontweight="bold")
    ax.grid(True,axis="y",linestyle="--",alpha=0.4)
    ax.spines[["top","right"]].set_visible(False); ax.legend(fontsize=10)
    ax.text(0.97,0.06,
            "H2-1 noise (2025-04-30)\n$p_2=1.05\\times10^{-3}$\n$p_{\\rm ro}=1.39\\times10^{-3}$",
            transform=ax.transAxes,fontsize=8,ha="right",va="bottom",fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.4",facecolor="lightyellow",edgecolor="#aaaaaa",alpha=0.88))

    ax2=axes[1]
    comps=[r"$\langle X\rangle$",r"$\langle Y\rangle$",r"$\langle Z\rangle$"]
    meas=list(results["bloch_measured"]); tgt=list(results["bloch_target"])
    xpos=np.arange(3); w=0.32
    b1=ax2.bar(xpos-w/2,tgt, w,color="#555555",alpha=0.75,edgecolor="white",label=r"Target $|\psi_M\rangle$")
    b2=ax2.bar(xpos+w/2,meas,w,color="#4c72b0",alpha=0.88,edgecolor="white",label=r"Reconstructed $\rho_Y$")
    for bars in [b1,b2]:
        for rect in bars:
            h=rect.get_height(); va="bottom" if h>=0 else "top"
            ax2.text(rect.get_x()+rect.get_width()/2, h+(0.02 if h>=0 else -0.02),
                     f"{h:.3f}",ha="center",va=va,fontsize=9)
    ax2.axhline(0,color="black",lw=0.7,alpha=0.5)
    ax2.set_xticks(xpos); ax2.set_xticklabels(comps,fontsize=13)
    ax2.set_ylabel("Bloch vector component",fontsize=12); ax2.set_ylim(-1.3,1.3)
    ax2.set_title("Bloch Vector: Target vs Reconstructed",fontsize=11,fontweight="bold")
    ax2.grid(True,axis="y",linestyle="--",alpha=0.4)
    ax2.spines[["top","right"]].set_visible(False); ax2.legend(fontsize=10)

    plt.tight_layout()
    fig.savefig("pctc_hardware_results.pdf",bbox_inches="tight",dpi=300)
    fig.savefig("pctc_hardware_results.png",bbox_inches="tight",dpi=300)
    print("Saved pctc_hardware_results.pdf / .png")
    plt.close()

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════
def run():
    print("="*60)
    print("  PCTC m=1 — Hardware Pipeline (H2-1)")
    print(f"  compile={BACKEND_COMPILE}  execute={BACKEND_EXECUTE}")
    print(f"  N_SHOTS={N_SHOTS}  OPT_LEVEL={OPT_LEVEL}")
    print("="*60)

    project = qnx.projects.get(name=PROJECT_NAME)
    print(f"\nProject: {project.annotations.name}")

    all_labels = ["Z","X","Y"] + SHADOW_GATES
    results = {
        "circuit":"PCTC","m":1,"backend":BACKEND_EXECUTE,
        "theta":THETA,"varphi":VARPHI,
        "n_shots":N_SHOTS,"bell_postselect":"00","shadow_gates":SHADOW_GATES,
    }

    # ── Cost estimate (compile first circuit only, rest are identical) ─────────
    print("\n── Pre-flight cost estimate ─────────────────────────────────")
    qc_probe = build_circuit("Z")
    tk_probe  = qiskit_to_tk(qc_probe)
    logical_ref = qnx.circuits.upload(
        name="PCTC_hw_Z_logical", circuit=tk_probe, project=project)
    compile_job = qnx.start_compile_job(
        programs=logical_ref,
        backend_config=qnx.QuantinuumConfig(device_name=BACKEND_COMPILE),
        optimisation_level=OPT_LEVEL,
        name="PCTC_hw_Z_compile_probe",
        project=project,
    )
    poll_job(compile_job, "compile:probe")
    compiled_probe = qnx.jobs.results(compile_job)[0].get_output().download_circuit()

    hqc_per, n1q, n2q, n_mid = estimate_hqc(compiled_probe, N_SHOTS)
    hqc_total = hqc_per * len(all_labels)
    print(f"  Compiled circuit: N_1q={n1q}  N_2q={n2q}  N_mid_meas={n_mid}")
    print(f"  HQC per circuit:  {hqc_per:.2f}")
    print(f"  HQC total ({len(all_labels)} circuits × {N_SHOTS} shots): {hqc_total:.2f}")
    print()
    confirm = input("  Proceed with hardware execution? [y/N]: ").strip().lower()
    if confirm != "y":
        print("  Aborted.")
        return None

    # ── Submit each circuit individually ──────────────────────────────────────
    raw_results = {}
    sel         = {}

    for label in all_labels:
        print(f"\n── Circuit [{label}] ──────────────────────────────────────────")
        qc     = build_circuit(label)
        result, compiled_tk = compile_and_execute(
            qc, project, f"PCTC_hw_{label}", N_SHOTS)
        bs     = extract_bitstrings(result)
        s      = post_select(bs, label=f"[{label}]")
        raw_results[label] = bs
        sel[label]         = s
        results[f"n_raw_{label}"] = len(bs)
        results[f"n_sel_{label}"] = len(s)
        # Save incrementally after each circuit
        results["bitstrings_sel"] = results.get("bitstrings_sel", {})
        results["bitstrings_sel"][label] = [list(b) for b in s]
        with open(RESULTS_FILE,"w") as f:
            json.dump(results,f,indent=2)
        print(f"  Saved incremental results → {RESULTS_FILE}")

    # ── Tomography fidelity ───────────────────────────────────────────────────
    print("\n── Tomography fidelity ──────────────────────────────────────")
    F_tomo,rho,bloch = tomography_fidelity(sel["Z"],sel["X"],sel["Y"])
    ci_lo,ci_hi,ci_std = bootstrap_tomo(sel["Z"],sel["X"],sel["Y"])
    tgt = target_bloch()
    print(f"  Bloch measured: rx={bloch[0]:.3f}  ry={bloch[1]:.3f}  rz={bloch[2]:.3f}")
    print(f"  Bloch target:   rx={tgt[0]:.3f}    ry={tgt[1]:.3f}    rz={tgt[2]:.3f}")
    print(f"  F_tomo = {F_tomo:.4f}   95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")

    # ── Shadows fidelity ──────────────────────────────────────────────────────
    print("\n── Classical shadows fidelity ───────────────────────────────")
    shadow_data = [(gate, b[2]) for gate in SHADOW_GATES for b in sel[gate]]
    print(f"  Total shadow shots (post-selected): {len(shadow_data)}")
    F_shad = shadow_fidelity_estimator(shadow_data)
    s_lo,s_hi,s_std = bootstrap_shadows(shadow_data)
    print(f"  F_shadows = {F_shad:.4f}   95% CI [{s_lo:.4f}, {s_hi:.4f}]")

    # ── Save final results ────────────────────────────────────────────────────
    results.update({
        "F_tomo":F_tomo,"F_tomo_ci":[ci_lo,ci_hi],"F_tomo_std":ci_std,
        "bloch_measured":list(bloch),"bloch_target":list(tgt),
        "rho_real":rho.real.tolist(),"rho_imag":rho.imag.tolist(),
        "F_shadows":F_shad,"F_shadows_ci":[s_lo,s_hi],"F_shadows_std":s_std,
        "n_shadow_total":len(shadow_data),
        "shadow_data":[[g,b] for g,b in shadow_data],
    })
    with open(RESULTS_FILE,"w") as f:
        json.dump(results,f,indent=2)
    print(f"\nSaved → {RESULTS_FILE}")

    plot_results(results)

    print("\n"+"="*60)
    print("  SUMMARY")
    print(f"  F (tomography)        = {F_tomo:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  F (classical shadows) = {F_shad:.4f}  [{s_lo:.4f}, {s_hi:.4f}]")
    print(f"  Post-selected shots   = {len(sel['Z'])} (Z basis)")
    print(f"  Shadow shots total    = {len(shadow_data)}")
    print("="*60)
    return results


if __name__ == "__main__":
    run()

  PCTC m=1 — Hardware Pipeline (H2-1)
  compile=H2-1E  execute=H2-1
  N_SHOTS=5000  OPT_LEVEL=3

Project: CTCs

── Pre-flight cost estimate ─────────────────────────────────
  Polling compile:probe... COMPLETED
  Compiled circuit: N_1q=29  N_2q=12  N_mid_meas=2
  HQC per circuit:  160.44
  HQC total (6 circuits × 5000 shots): 962.61


── Circuit [Z] ──────────────────────────────────────────
  Polling compile:PCTC_hw_Z. COMPLETED
  Polling execute:PCTC_hw_Z.. ERROR
  error_detail: Job cost exceeds allowed cost
  message: Program has encountered an error


RuntimeError: execute:PCTC_hw_Z failed: ERROR

In [6]:
import qnexus as qnx

project = qnx.projects.get(name="CTCs")

# Check account/project cost limits
try:
    print(qnx.backends.get_all())
except: pass

# Try a tiny test job to find the per-job limit
# Submit just 1 circuit with 10 shots to see if it passes
from qiskit import QuantumCircuit
from pytket.extensions.qiskit import qiskit_to_tk

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
tk = qiskit_to_tk(qc)

ref = qnx.circuits.upload(name="cost_probe", circuit=tk, project=project)
job = qnx.start_execute_job(
    programs=ref,
    backend_config=qnx.QuantinuumConfig(device_name="H2-1"),
    n_shots=10,
    name="cost_probe_execute",
    project=project,
)
import time
while True:
    s = qnx.jobs.status(job)
    if s.status.value in ("COMPLETED", "ERROR", "CANCELLED"):
        break
    time.sleep(5)
print(f"Status: {s.status.value}")
print(f"Full status: {s}")

Status: ERROR
Full status: JobStatus(status=<JobStatusEnum.ERROR: 'ERROR'>, message='Circuit has encountered an error.', error_detail="<class 'pytket.backends.backend_exceptions.CircuitNotValidError'>: Circuit with index 0 in submitted does not satisfy GateSetPredicate:{ JobShotNum RNGNum RNGIndex RNGBound RNGSeed ClExpr Rz WASM Reset CopyBits PhasedX ExplicitModifier Measure Barrier TK2 ZZMax SetBits RangePredicate ZZPhase ExplicitPredicate MultiBit } (try compiling with backend.get_compiled_circuits first).", completed_time=None, queued_time=None, submitted_time=datetime.datetime(2026, 2, 26, 22, 49, 20, 791439, tzinfo=datetime.timezone.utc), running_time=None, cancelled_time=None, error_time=datetime.datetime(2026, 2, 26, 22, 49, 21, 421852, tzinfo=datetime.timezone.utc), queue_position=None, cost=None)


In [17]:
import qnexus as qnx

# See all available backend config types
print(dir(qnx))

# Check what the Helios device entry looks like
backends = qnx.devices.get_all()
# for b in backends:
#     if "" in b.device_name:
#         print(b)
#         print(type(b.stored_backend_info))
#         print(b.stored_backend_info.name)

['AerConfig', 'AerStateConfig', 'AerUnitaryConfig', 'BackendConfig', 'BraketConfig', 'IBMQConfig', 'IBMQEmulatorConfig', 'QuantinuumConfig', 'QulacsConfig', 'SeleneConfig', 'SelenePlusConfig', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'auth', 'circuits', 'client', 'compile', 'config', 'context', 'credentials', 'devices', 'exceptions', 'execute', 'filesystem', 'gpu_decoder_configs', 'hugr', 'jobs', 'login', 'login_with_credentials', 'logout', 'models', 'nest_asyncio', 'projects', 'qir', 'quotas', 'roles', 'start_compile_job', 'start_execute_job', 'teams', 'users', 'warnings', 'wasm_modules']
